# 4. MCoverDomainboundary

Part of the **[Fig. 5 chapter](fig5.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}clustering/merged/group_meta.tsv'`  ·  _metadata_
- `f'{indir}merged_allc/cluster_donor.mcds'`  ·  _sc/pseudobulk mC (allc)_
- `'mCG_distribution/L1_global_mCG.tsv.gz'`  ·  _table_
- `'domain/L1_raw.boundary.h5ad'`  ·  _domain/boundary_
- `'domain/domain_bound_diff0.joblib'`  ·  _domain/boundary_
- `'mCG_distribution/L1_chrom5k_mCG.hdf'`  ·  _other_
- `'mCH_distribution/L1_chrom5k_mCH.hdf'`  ·  _other_


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob
from scipy.stats import pearsonr
import joblib

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances, roc_auc_score

from ALLCools.mcds import MCDS
import cooler

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
chrom_size_path = '/large_experiments/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
# group_meta = group_meta[['L2_any', 'L1', 'count']]
group_meta['L1_annot'] = group_meta['L1_annot'].str.replace(' ','-').str.replace('/','_')
annot2L1 = group_meta[['L1','L1_annot']].set_index('L1_annot')['L1'].to_dict()
L1annot = group_meta[['L1','L1_annot']].set_index('L1')['L1_annot'].to_dict()


## Get mCG level

In [ ]:
mcds = MCDS.open(f'{indir}merged_allc/cluster_donor.mcds', var_dim='chrom1k')
mcds

In [ ]:
bin_df = mcds[['chrom1k_chrom', 'chrom1k_start', 'chrom1k_end']].to_pandas()
bin_df['chrom5k'] = bin_df['chrom1k_chrom'].astype(str) + '_' + (bin_df['chrom1k_start'] // 5000).astype(str)
bin_df

In [ ]:
mcds = mcds.assign_coords(L1=('cell', group_meta.loc[mcds.get_index('cell'), 'L1']))
mcds = mcds.groupby('L1').sum()


In [ ]:
# mcds = mcds.assign_coords(chrom10k=('chrom1k', bin_df['chrom1k']))
mcds = mcds.sel({'mc_type':'CGN'})
mcds = MCDS(mcds, obs_dim='L1', var_dim='chrom1k')
mcds

In [ ]:
exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [ ]:
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='L1').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, log_scale=(True, False), ax=ax)
# ax.set_yscale('log')

In [ ]:
mcds = mcds.sel({'chrom1k':cov.index[cov>50]})

black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)


In [ ]:
global_cg = mcds['chrom1k_da'].sum(dim='chrom1k').to_pandas()
global_cg = global_cg['mc'] / global_cg['cov']

In [ ]:
global_cg.to_csv('mCG_distribution/L1_global_mCG.tsv.gz', sep='\t', header=False)


In [ ]:
global_cg = pd.read_csv('mCG_distribution/L1_global_mCG.tsv.gz', sep='\t', header=None, index_col=0)


In [ ]:
mc = mcds.sel({'count_type':'mc'})['chrom1k_da'].to_pandas()
cov = mcds.sel({'count_type':'cov'})['chrom1k_da'].to_pandas()

In [ ]:
bin_df = bin_df.loc[mc.columns]
mc = mc.T.groupby(bin_df['chrom5k']).sum()
cov = cov.T.groupby(bin_df['chrom5k']).sum()


In [ ]:
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
mcg = calculate_posterior_mc_frac(mc.T.values, cov.T.values, normalize_per_cell=False)
mcg = pd.DataFrame(mcg, index=mc.columns, columns=mc.index)


In [ ]:
mcg.to_hdf('mCG_distribution/L1_chrom5k_mCG.hdf', key='data')


In [ ]:
adata = anndata.read_h5ad('domain/L1_raw.boundary.h5ad')


In [ ]:
bin_domain = adata.var.copy()
bin_domain.index = bin_domain.index.astype(int)
chrom_offset = {c: bin_domain[bin_domain['chrom'] == c].index[0] for c in chrom_sizes.index}


In [ ]:
from sklearn.metrics import roc_auc_score

domain = {}
ws = 5
for i,ct in enumerate(adata.obs.index):
    domaintmp = adata.X.indices[adata.X.indptr[i]:adata.X.indptr[i+1]]
    chrom = bin_domain.loc[domaintmp, 'chrom'].values
    group_domain_tmp = {'real':{}, 'shuffle':{}}
    for c in chrom_sizes.index:
        bound_ct_chrom = domaintmp[chrom==c] - chrom_offset[c]
        
        ## filter nearby boundaries
        # diff = np.abs(np.diff(bound_ct_chrom))
        # boundfilter = np.ones(len(bound_ct_chrom)).astype(bool)
        # boundfilter[1:] = np.logical_and(boundfilter[1:], diff>10)
        # boundfilter[:-1] = np.logical_and(boundfilter[:-1], diff>10)
        # group_domain_tmp['real'][c] = bound_ct_chrom[boundfilter] * 25000
        group_domain_tmp['real'][c] = bound_ct_chrom * 25000
        
        ## random background boudaries
        bound_ct_chrom_flank = np.zeros((bin_domain['chrom']==c).sum())
        bound_ct_chrom_flank[bound_ct_chrom] = 1
        bound_ct_chrom_flank = np.convolve(bound_ct_chrom_flank, np.ones(ws*2+1))[ws:-ws]
        bound_ct_chrom_flank = np.where(bound_ct_chrom_flank==0)[0]
        bound_ct_chrom_flank = np.random.choice(bound_ct_chrom_flank, len(bound_ct_chrom), False)
        bound_ct_chrom_flank = np.sort(bound_ct_chrom_flank)
        
        # diff = np.abs(np.diff(bound_ct_chrom_flank))
        # boundfilter = np.ones(len(bound_ct_chrom_flank)).astype(bool)
        # boundfilter[1:] = np.logical_and(boundfilter[1:], diff>10)
        # boundfilter[:-1] = np.logical_and(boundfilter[:-1], diff>10)
        # group_domain_tmp['shuffle'][c] = bound_ct_chrom_flank[boundfilter] * 25000
        group_domain_tmp['shuffle'][c] = bound_ct_chrom_flank * 25000

    domain[ct] = group_domain_tmp.copy()
    print(ct)
    

In [ ]:
joblib.dump(domain, 'domain/domain_bound_diff0.joblib', compress=5)


In [ ]:
domain = joblib.load('domain/domain_bound_diff0.joblib')


In [ ]:
mcg = pd.read_hdf('mCG_distribution/L1_chrom5k_mCG.hdf', key='data')


In [ ]:
bin_df = pd.DataFrame(index=mcg.columns)
bin_df['chrom'] = bin_df.index.str.split('_').str[0]
bin_df['start'] = bin_df.index.str.split('_').str[1].astype(int) * 5000
bin_df

In [ ]:
for ct in ['c2','c7','c1','c22','c10','c21','c14','c20']:
    mcg_ct = mcg.loc[ct].groupby(bin_df['chrom'])
    bound_data = {'real':[], 'shuffle':[]}
    for c,mcg_ct_chrom in mcg_ct:
        idx = mcg_ct_chrom.index.str.split('_').str[1].astype(int) * 5000
        for key in ['real', 'shuffle']:
            tmp = np.zeros((len(domain[ct][key][c]), 100))
            boundfilter = np.ones(len(domain[ct][key][c])).astype(bool)
            for i,xx in enumerate(domain[ct][key][c]):
                lbin = (idx >= (xx - 250000)) & (idx < xx)
                rbin = (idx < (xx + 250000)) & (idx >= xx)
                selb = lbin | rbin
                mcg_tmp = mcg_ct_chrom[selb]
                if mcg_tmp.shape[0]<30:
                    boundfilter[i] = False
                idxtmp = idx[selb]
                tmp[i] = mcg_tmp.mean()
                tmp[i, (idxtmp-xx)//5000 + 50] = mcg_tmp.values
            bound_data[key].append(tmp[boundfilter])

    for key in ['real', 'shuffle']:
        tmp = bound_data[key]
        tmp = np.concatenate(tmp, axis=0)
        selb = tmp[:, :50].mean(axis=1) < tmp[:, 50:].mean(axis=1)
        tmp[selb] = tmp[selb, ::-1]
        idx = np.argsort(tmp[:, :50].mean(axis=1) - tmp[:, 50:].mean(axis=1))
        bound_data[key] = tmp[idx]
        
    fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)
    fig.suptitle(L1annot[ct], fontsize=12)
    for i,key in enumerate(['real', 'shuffle']):
        ax = axes[i]
        data = bound_data[key].copy()
        if key=='real':
            vmin, vmax = np.percentile(data, [5, 98])
        ax.imshow(data, cmap='Greens_r', aspect='auto', vmin=vmin, vmax=vmax, rasterized=True)
        ax.set_xticks([-0.5, 49.5, 99.5])
        ax.set_yticks([])
        ax.set_ylim([-0.5, len(data)-0.5])
        ax.set_xticklabels(['-250k', ['Boundary', 'Random'][i], '+250k'])
        ax.set_ylabel(f'{data.shape[0]} Domain Boundaries')
        


In [ ]:
for ct in ['c2','c7','c1','c22','c10','c21','c14','c20']:
    mcg_ct = mcg.loc[ct].groupby(bin_df['chrom'])
    bound_data = {'real':[], 'shuffle':[]}
    for c,mcg_ct_chrom in mcg_ct:
        idx = mcg_ct_chrom.index.str.split('_').str[1].astype(int) * 5000
        for key in ['real', 'shuffle']:
            tmp = np.zeros((len(domain[ct][key][c]), 100))
            boundfilter = np.ones(len(domain[ct][key][c])).astype(bool)
            for i,xx in enumerate(domain[ct][key][c]):
                lbin = (idx >= (xx - 250000)) & (idx < xx)
                rbin = (idx < (xx + 250000)) & (idx >= xx)
                selb = lbin | rbin
                mcg_tmp = mcg_ct_chrom[selb]
                if mcg_tmp.shape[0]<30:
                    boundfilter[i] = False
                idxtmp = idx[selb]
                tmp[i] = mcg_tmp.mean()
                tmp[i, (idxtmp-xx)//5000 + 50] = mcg_tmp.values
            bound_data[key].append(tmp[boundfilter])

    for key in ['real', 'shuffle']:
        tmp = bound_data[key]
        tmp = np.concatenate(tmp, axis=0)
        selb = tmp[:, :50].mean(axis=1) < tmp[:, 50:].mean(axis=1)
        tmp[selb] = tmp[selb, ::-1]
        idx = np.argsort(tmp[:, :50].mean(axis=1) - tmp[:, 50:].mean(axis=1))
        bound_data[key] = tmp[idx]
        
    fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)
    fig.suptitle(L1annot[ct], fontsize=12)
    for i,key in enumerate(['real', 'shuffle']):
        ax = axes[i]
        data = bound_data[key].copy()
        if key=='real':
            vmin, vmax = np.percentile(data, [5, 98])
        ax.imshow(data, cmap='Greens_r', aspect='auto', vmin=vmin, vmax=vmax, rasterized=True)
        ax.set_xticks([-0.5, 49.5, 99.5])
        ax.set_yticks([])
        ax.set_ylim([-0.5, len(data)-0.5])
        ax.set_xticklabels(['-250k', ['Boundary', 'Random'][i], '+250k'])
        ax.set_ylabel(f'{data.shape[0]} Domain Boundaries')
        


In [ ]:
# for ct in ['c2','c7','c1','c22','c10','c21','c14','c20']:
#     bound = {}
#     domaintmp = domain[ct]
#     chrom = bin_tmp.loc[domaintmp, 'chrom'].values
#     for c in chrom_sizes.index:
#         bound_ct_chrom = domaintmp[chrom==c]
#         diff = np.abs(np.diff(bound_ct_chrom))
#         boundfilter = np.ones(len(bound_ct_chrom)).astype(bool)
#         boundfilter[1:] = np.logical_and(boundfilter[1:], diff>5)
#         boundfilter[:-1] = np.logical_and(boundfilter[:-1], diff>5)
#         bound[c] = (bound_ct_chrom[boundfilter] - chrom_offset[c]) * 25000
        
#     mcg_ct = mcg.loc[ct].groupby(bin_df['chrom'])
#     bound_data = []
#     for c,mcg_ct_chrom in mcg_ct:
#         idx = mcg_ct_chrom.index.str.split('_').str[1].astype(int) * 5000
#         tmp = np.zeros((len(bound[c]), 100))
#         boundfilter = np.ones(len(bound[c])).astype(bool)
#         for i,xx in enumerate(bound[c]):
#             lbin = (idx >= (xx - 250000)) & (idx < xx)
#             rbin = (idx < (xx + 250000)) & (idx >= xx)
#             selb = lbin | rbin
#             mcg_tmp = mcg_ct_chrom[selb]
#             if mcg_tmp.shape[0]<30:
#                 boundfilter[i] = False
#             idxtmp = idx[selb]
#             tmp[i] = mcg_tmp.mean()
#             tmp[i, (idxtmp-xx)//5000 + 50] = mcg_tmp.values
#         bound_data.append(tmp[boundfilter])

#     bound_data = np.concatenate(bound_data, axis=0)
#     selb = bound_data[:, :50].mean(axis=1) < bound_data[:, 50:].mean(axis=1)
#     bound_data[selb] = bound_data[selb, ::-1]
#     idx = np.argsort(bound_data[:, :50].mean(axis=1) - bound_data[:, 50:].mean(axis=1))[::-1]
#     bound_data = bound_data[idx]
    
#     fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)
#     fig.suptitle(L1annot[ct], fontsize=12)
#     vmin, vmax = np.percentile(bound_data, [5, 98])
#     for i,ax in enumerate(axes):
#         ax.imshow(bound_data, cmap='Greens_r', aspect='auto', interpolation=['none', 'antialiased'][i], vmin=vmin, vmax=vmax)
#         ax.set_xticks([-0.5, 49.5, 99.5])
#         ax.set_yticks([])
#         ax.set_xticklabels(['-250k', 'Domain Boundary', '+250k'])
#         ax.set_ylabel(f'{bound_data.shape[0]} Domain Boundaries')

    

## Get mCH level

In [ ]:
mcds = MCDS.open(f'{indir}merged_allc/cluster_donor.mcds', var_dim='chrom1k')
mcds

In [ ]:
bin_df = mcds[['chrom1k_chrom', 'chrom1k_start', 'chrom1k_end']].to_pandas()
bin_df['chrom5k'] = bin_df['chrom1k_chrom'].astype(str) + '_' + (bin_df['chrom1k_start'] // 5000).astype(str)
bin_df

In [ ]:
mcds = mcds.assign_coords(L1=('cell', group_meta.loc[mcds.get_index('cell'), 'L1']))
mcds = mcds.groupby('L1').sum()


In [ ]:
# mcds = mcds.assign_coords(chrom10k=('chrom1k', bin_df['chrom1k']))
mcds = mcds.sel({'mc_type':['CAN','CCN','CTN']}).sum(dim='mc_type')
mcds = MCDS(mcds, obs_dim='L1', var_dim='chrom1k')
mcds

In [ ]:
exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [ ]:
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='L1').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, log_scale=(True, False), ax=ax)
# ax.set_yscale('log')

In [ ]:
mcds = mcds.sel({'chrom1k':cov.index[cov>500]})

black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)


In [ ]:
global_cg = mcds['chrom1k_da'].sum(dim='chrom1k').to_pandas()
global_cg = global_cg['mc'] / global_cg['cov']

In [ ]:
global_cg.to_csv('mCH_distribution/L1_global_mCH.tsv.gz', sep='\t', header=False)


In [ ]:
mcds

In [ ]:
mc = mcds.sel({'count_type':'mc'})['chrom1k_da'].to_pandas()
cov = mcds.sel({'count_type':'cov'})['chrom1k_da'].to_pandas()

In [ ]:
bin_df = bin_df.loc[mc.columns]
mc = mc.T.groupby(bin_df['chrom5k']).sum()
cov = cov.T.groupby(bin_df['chrom5k']).sum()


In [ ]:
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
mcg = calculate_posterior_mc_frac(mc.T.values, cov.T.values, normalize_per_cell=False)
mcg = pd.DataFrame(mcg, index=mc.columns, columns=mc.index)


In [ ]:
mcg.to_hdf('mCH_distribution/L1_chrom5k_mCH.hdf', key='data')


In [ ]:
mcg = pd.read_hdf('mCH_distribution/L1_chrom5k_mCH.hdf', key='data')


In [ ]:
bin_df = pd.DataFrame(index=mcg.columns)
bin_df['chrom'] = bin_df.index.str.split('_').str[0]
bin_df['start'] = bin_df.index.str.split('_').str[1].astype(int) * 5000
bin_df

In [ ]:
for ct in ['c2','c7','c1','c22','c10','c21','c14','c20']:
    mcg_ct = mcg.loc[ct].groupby(bin_df['chrom'])
    bound_data = {'real':[], 'shuffle':[]}
    for c,mcg_ct_chrom in mcg_ct:
        idx = mcg_ct_chrom.index.str.split('_').str[1].astype(int) * 5000
        for key in ['real', 'shuffle']:
            tmp = np.zeros((len(domain[ct][key][c]), 100))
            boundfilter = np.ones(len(domain[ct][key][c])).astype(bool)
            for i,xx in enumerate(domain[ct][key][c]):
                lbin = (idx >= (xx - 250000)) & (idx < xx)
                rbin = (idx < (xx + 250000)) & (idx >= xx)
                selb = lbin | rbin
                mcg_tmp = mcg_ct_chrom[selb]
                if mcg_tmp.shape[0]<30:
                    boundfilter[i] = False
                idxtmp = idx[selb]
                tmp[i] = mcg_tmp.mean()
                tmp[i, (idxtmp-xx)//5000 + 50] = mcg_tmp.values
            bound_data[key].append(tmp[boundfilter])

    for key in ['real', 'shuffle']:
        tmp = bound_data[key]
        tmp = np.concatenate(tmp, axis=0)
        selb = tmp[:, :50].mean(axis=1) < tmp[:, 50:].mean(axis=1)
        tmp[selb] = tmp[selb, ::-1]
        idx = np.argsort(tmp[:, :50].mean(axis=1) - tmp[:, 50:].mean(axis=1))
        bound_data[key] = tmp[idx]
        
    fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)
    fig.suptitle(L1annot[ct], fontsize=12)
    for i,key in enumerate(['real', 'shuffle']):
        ax = axes[i]
        data = bound_data[key].copy()
        if key=='real':
            vmin, vmax = np.percentile(data, [5, 98])
        ax.imshow(data, cmap='Purples_r', aspect='auto', vmin=vmin, vmax=vmax, rasterized=True)
        ax.set_xticks([-0.5, 49.5, 99.5])
        ax.set_yticks([])
        ax.set_ylim([-0.5, len(data)-0.5])
        ax.set_xticklabels(['-250k', ['Boundary', 'Random'][i], '+250k'])
        ax.set_ylabel(f'{data.shape[0]} Domain Boundaries')
        


In [ ]:
for ct in ['c2','c7','c1','c22','c10','c21','c14','c20']:
    mcg_ct = mcg.loc[ct].groupby(bin_df['chrom'])
    bound_data = {'real':[], 'shuffle':[]}
    for c,mcg_ct_chrom in mcg_ct:
        idx = mcg_ct_chrom.index.str.split('_').str[1].astype(int) * 5000
        for key in ['real', 'shuffle']:
            tmp = np.zeros((len(domain[ct][key][c]), 100))
            boundfilter = np.ones(len(domain[ct][key][c])).astype(bool)
            for i,xx in enumerate(domain[ct][key][c]):
                lbin = (idx >= (xx - 250000)) & (idx < xx)
                rbin = (idx < (xx + 250000)) & (idx >= xx)
                selb = lbin | rbin
                mcg_tmp = mcg_ct_chrom[selb]
                if mcg_tmp.shape[0]<30:
                    boundfilter[i] = False
                idxtmp = idx[selb]
                tmp[i] = mcg_tmp.mean()
                tmp[i, (idxtmp-xx)//5000 + 50] = mcg_tmp.values
            bound_data[key].append(tmp[boundfilter])

    for key in ['real', 'shuffle']:
        tmp = bound_data[key]
        tmp = np.concatenate(tmp, axis=0)
        selb = tmp[:, :50].mean(axis=1) < tmp[:, 50:].mean(axis=1)
        tmp[selb] = tmp[selb, ::-1]
        idx = np.argsort(tmp[:, :50].mean(axis=1) - tmp[:, 50:].mean(axis=1))
        bound_data[key] = tmp[idx]
        
    fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)
    fig.suptitle(L1annot[ct], fontsize=12)
    for i,key in enumerate(['real', 'shuffle']):
        ax = axes[i]
        data = bound_data[key].copy()
        if key=='real':
            vmin, vmax = np.percentile(data, [5, 98])
        ax.imshow(data, cmap='Purples_r', aspect='auto', vmin=vmin, vmax=vmax, rasterized=True)
        ax.set_xticks([-0.5, 49.5, 99.5])
        ax.set_yticks([])
        ax.set_ylim([-0.5, len(data)-0.5])
        ax.set_xticklabels(['-250k', ['Boundary', 'Random'][i], '+250k'])
        ax.set_ylabel(f'{data.shape[0]} Domain Boundaries')
        


In [ ]:
# for ct in ['c2','c7','c1','c22','c10','c21','c14','c20']:
#     bound = {}
#     domaintmp = domain[ct]
#     chrom = bin_tmp.loc[domaintmp, 'chrom'].values
#     for c in chrom_sizes.index:
#         bound_ct_chrom = domaintmp[chrom==c]
#         diff = np.abs(np.diff(bound_ct_chrom))
#         boundfilter = np.ones(len(bound_ct_chrom)).astype(bool)
#         boundfilter[1:] = np.logical_and(boundfilter[1:], diff>5)
#         boundfilter[:-1] = np.logical_and(boundfilter[:-1], diff>5)
#         bound[c] = (bound_ct_chrom[boundfilter] - chrom_offset[c]) * 25000
        
#     mcg_ct = mcg.loc[ct].groupby(bin_df['chrom'])
#     bound_data = []
#     for c,mcg_ct_chrom in mcg_ct:
#         idx = mcg_ct_chrom.index.str.split('_').str[1].astype(int) * 5000
#         tmp = np.zeros((len(bound[c]), 100))
#         boundfilter = np.ones(len(bound[c])).astype(bool)
#         for i,xx in enumerate(bound[c]):
#             lbin = (idx >= (xx - 250000)) & (idx < xx)
#             rbin = (idx < (xx + 250000)) & (idx >= xx)
#             selb = lbin | rbin
#             mcg_tmp = mcg_ct_chrom[selb]
#             if mcg_tmp.shape[0]<30:
#                 boundfilter[i] = False
#             idxtmp = idx[selb]
#             tmp[i] = mcg_tmp.mean()
#             tmp[i, (idxtmp-xx)//5000 + 50] = mcg_tmp.values
#         bound_data.append(tmp[boundfilter])

#     bound_data = np.concatenate(bound_data, axis=0)
#     selb = bound_data[:, :50].mean(axis=1) < bound_data[:, 50:].mean(axis=1)
#     bound_data[selb] = bound_data[selb, ::-1]
#     idx = np.argsort(bound_data[:, :50].mean(axis=1) - bound_data[:, 50:].mean(axis=1))[::-1]
#     bound_data = bound_data[idx]
    
#     fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=300)
#     fig.suptitle(L1annot[ct], fontsize=12)
#     vmin, vmax = np.percentile(bound_data, [5, 98])
#     for i,ax in enumerate(axes):
#         ax.imshow(bound_data, cmap='Purples_r', aspect='auto', interpolation=['none', 'antialiased'][i], vmin=vmin, vmax=vmax)
#         ax.set_xticks([-0.5, 49.5, 99.5])
#         ax.set_yticks([])
#         ax.set_xticklabels(['-250k', 'Domain Boundary', '+250k'])
#         ax.set_ylabel(f'{bound_data.shape[0]} Domain Boundaries')

    